In [2]:
%%writefile API/app.py
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
import pickle
import numpy as np
import re

from services.LSTM_service import LSTMService

MAX_LEN = 100
MAX_WORDS = 10000
PATH_MODEL = "./Modeles/LSTM/modele_lstm" 
PATH_TOKENIZER = "./Modeles/LSTM/tokenizer.pickle"

model = LSTMService(PATH_MODEL, PATH_TOKENIZER)

# --- 3. DÉFINITION DE L'APPLICATION ---
app = FastAPI(
    title="Air Paradis Sentiment API", 
    description="API de prédiction de sentiment (LSTM)",
    version="1.0.0"
)

# On définit le format attendu des données (Un JSON avec un champ "text")
class TweetInput(BaseModel):
    text: str

# Fonction de nettoyage (Doit être la même que celle de ton entraînement !)
def nettoyer_texte(texte):
    texte = texte.lower()
    texte = re.sub(r'<.*?>', '', texte)
    texte = re.sub(r'https?://\S+|www\.\S+', '', texte)
    texte = re.sub(r'[^a-zA-Z\s]', '', texte) # On garde lettres et espaces
    return texte

# --- 4. LES ENDPOINTS (Les routes) ---

@app.get("/")
def read_root():
    return {"message": "Bienvenue sur l'API Air Paradis. Utilisez /predict pour analyser un tweet."}

@app.post("/predict")
def predict_sentiment(input_data: TweetInput):
    try:
        # A. Récupération et nettoyage
        raw_text = input_data.text

        # D. Prédiction
        # Le modèle renvoie un tableau (ex: [[0.85]]), on prend la valeur float
        prediction = model.predict(raw_text)
        
        # E. Interprétation
        sentiment_label = "Positif" if prediction > 0.5 else "Négatif"
        
        # F. Réponse JSON
        return {
            "tweet_original": raw_text,
            "sentiment": sentiment_label,
            "score": float(prediction) # On convertit en float python standard
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Writing API/app.py


FileNotFoundError: [Errno 2] No such file or directory: 'API/app.py'

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Toujours utile sur Mac M1

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from contextlib import asynccontextmanager
from core.lstm_service import LSTMService

# --- CONFIGURATION ---
# Mets ici le chemin vers le dossier créé par model.export()
PATH_MODEL = "../Modeles/LSTM/modele_lstm" 
PATH_TOKENIZER = "../Modeles/LSTM/tokenizer.pickle"

service = None

# --- DÉMARRAGE ---
@asynccontextmanager
async def lifespan(app: FastAPI):
    global service
    try:
        service = LSTMService(PATH_MODEL, PATH_TOKENIZER)
        print("✅ Modèle chargé et prêt !")
    except Exception as e:
        print(f"❌ Erreur critique : {e}")
    yield
    print("Arrêt du service.")

app = FastAPI(title="API Prédiction LSTM", lifespan=lifespan)

class TextRequest(BaseModel):
    text: str

@app.post("/predict")
def predict(req: TextRequest):
    if not service:
        raise HTTPException(503, "Le modèle n'est pas chargé")
    
    try:
        return service.predict(req.text)
    except Exception as e:
        raise HTTPException(500, f"Erreur de prédiction : {str(e)}")